Cuaderno 2: Entrenamiento del Modelo

En esta primera celda, importamos las librerías necesarias para el preprocesamiento matemático, la creación del modelo predictivo y la exportación del mismo. También configuramos las rutas absolutas para asegurar que el cuaderno encuentre el archivo .parquet que generamos en el paso anterior y sepa dónde guardar el modelo final.

In [5]:
%pip install pandas xgboost scikit-learn joblib

   ---------------------------------------- 0.0/10.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/10.0 MB ? eta -:--:--
   ---------------------------------------- 0.1/10.0 MB 1.8 MB/s eta 0:00:06
   - -------------------------------------- 0.4/10.0 MB 3.3 MB/s eta 0:00:03
   --- ------------------------------------ 0.8/10.0 MB 4.3 MB/s eta 0:00:03
   ---- ----------------------------------- 1.1/10.0 MB 5.5 MB/s eta 0:00:02
   ----- ---------------------------------- 1.4/10.0 MB 5.6 MB/s eta 0:00:02
   ------- -------------------------------- 1.9/10.0 MB 6.0 MB/s eta 0:00:02
   -------- ------------------------------- 2.1/10.0 MB 6.3 MB/s eta 0:00:02
   ---------- ----------------------------- 2.6/10.0 MB 6.5 MB/s eta 0:00:02
   ----------- ---------------------------- 2.9/10.0 MB 6.7 MB/s eta 0:00:02
   ------------ --------------------------- 3.1/10.0 MB 6.4 MB/s eta 0:00:02
   -------------- ------------------------- 3.5/10.0 MB 6.8 MB/s eta 0:00:01
   ----------


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
import os
import pandas as pd
import joblib
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

print("Importando librerías de Machine Learning (Pandas, XGBoost, Scikit-Learn)...")
print("Librerías importadas correctamente.")

# Configuración de directorios
BASE_DIR = os.path.abspath('')
DATA_DIR = os.path.join(BASE_DIR, "data", "raw")
MODELS_DIR = os.path.join(BASE_DIR, "models")

# Creación de carpeta models si no existe
os.makedirs(MODELS_DIR, exist_ok=True)
print(f"Directorio de modelos configurado en: {MODELS_DIR}")

SYMBOL = "EURUSD"
RUTA_DATOS = os.path.join(DATA_DIR, f"{SYMBOL}_M5.parquet")
RUTA_MODELO = os.path.join(MODELS_DIR, f"{SYMBOL}_M5_scalper.joblib")

print(f"Símbolo a modelar: {SYMBOL}")
print(f"Ruta de lectura de datos: {RUTA_DATOS}")

Importando librerías de Machine Learning (Pandas, XGBoost, Scikit-Learn)...
Librerías importadas correctamente.
Directorio de modelos configurado en: c:\Users\Sebas Uni\Desktop\AITradingIntegrated\models
Símbolo a modelar: EURUSD
Ruta de lectura de datos: c:\Users\Sebas Uni\Desktop\AITradingIntegrated\data\raw\EURUSD_M5.parquet


Ahora procedemos a cargar los datos procesados. Una vez cargados, filtramos el histórico para trabajar con el periodo establecido (2023-2025). Luego, dividimos los datos en dos bloques: uno para entrenar al modelo (In-Sample) y otro para validar su comportamiento con datos que nunca ha visto (Out-of-Sample).

In [8]:
%pip install pyarrow

   ---------------------------------------- 0.0/27.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/27.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/27.8 MB 330.3 kB/s eta 0:01:25
   ---------------------------------------- 0.1/27.8 MB 550.5 kB/s eta 0:00:51
   ---------------------------------------- 0.2/27.8 MB 1.0 MB/s eta 0:00:28
   ---------------------------------------- 0.3/27.8 MB 1.5 MB/s eta 0:00:19
    --------------------------------------- 0.6/27.8 MB 2.5 MB/s eta 0:00:11
   - -------------------------------------- 0.8/27.8 MB 3.1 MB/s eta 0:00:09
   - -------------------------------------- 1.2/27.8 MB 3.7 MB/s eta 0:00:08
   - -------------------------------------- 1.3/27.8 MB 3.7 MB/s eta 0:00:08
   -- ------------------------------------- 1.6/27.8 MB 4.1 MB/s eta 0:00:07
   -- ------------------------------------- 1.9/27.8 MB 4.3 MB/s eta 0:00:07
   --- ------------------------------------ 2.1/27.8 MB 4.4 MB/s eta 0:00:06
   --- --


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
%pip install --upgrade pyarrow pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
%pip install fastparquet

   ---------------------------------------- 0.0/698.1 kB ? eta -:--:--
    --------------------------------------- 10.2/698.1 kB ? eta -:--:--
   ----- ---------------------------------- 92.2/698.1 kB 1.3 MB/s eta 0:00:01
   ------------- -------------------------- 235.5/698.1 kB 2.4 MB/s eta 0:00:01
   ---------------------------- ----------- 501.8/698.1 kB 3.5 MB/s eta 0:00:01
   ---------------------------------------- 698.1/698.1 kB 4.0 MB/s eta 0:00:00
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   -------- ------------------------------- 0.3/1.7 MB 10.9 MB/s eta 0:00:01
   ------------------- -------------------- 0.8/1.7 MB 10.4 MB/s eta 0:00:01
   --------------------------- ------------ 1.2/1.7 MB 10.8 MB/s eta 0:00:01
   ------------------------------------ --- 1.6/1.7 MB 10.0 MB/s eta 0:00:01
   ---------------------------------------- 1.7/1.7 MB 9.1 MB/s eta 0:00:00
   ---------------------------------------- 0.0/203.9 kB ? eta -:--:--
   -----------


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
print("Iniciando carga de datos Parquet...")
if not os.path.exists(RUTA_DATOS):
    print("ERROR CRÍTICO: No se encontró el archivo Parquet. Ejecuta el Cuaderno 1 primero.")
else:
    df_feat = pd.read_parquet(RUTA_DATOS, engine='fastparquet')
    print(f"Datos cargados exitosamente. Filas totales disponibles: {len(df_feat)}")

    print("Aplicando recorte temporal global: 01-01-2023 al 31-12-2025...")
    df_feat = df_feat.loc['2023-01-01':'2025-12-31']
    print(f"Filas resultantes tras el recorte temporal: {len(df_feat)}")

    print("Dividiendo dataset: In-Sample (Entrenamiento) y Out-of-Sample (Validación)...")
    df_is = df_feat.loc['2023-01-01':'2024-12-31']
    df_oos = df_feat.loc['2025-01-01':'2025-12-31']

    print(f"  -> Conjunto de Entrenamiento (IS): {len(df_is)} filas.")
    print(f"  -> Conjunto de Validación (OOS): {len(df_oos)} filas.")

Iniciando carga de datos Parquet...
Datos cargados exitosamente. Filas totales disponibles: 208631
Aplicando recorte temporal global: 01-01-2023 al 31-12-2025...
Filas resultantes tras el recorte temporal: 93326
Dividiendo dataset: In-Sample (Entrenamiento) y Out-of-Sample (Validación)...
  -> Conjunto de Entrenamiento (IS): 62246 filas.
  -> Conjunto de Validación (OOS): 31080 filas.


En esta fase, extraemos las características (features) y la variable objetivo (target) y las convertimos en matrices numéricas que el algoritmo puede procesar. Como en el trading suele haber un desbalance (hay menos operaciones exitosas que velas en el gráfico), calculamos un peso matemático (scale_pos_weight) para equilibrar el aprendizaje del modelo.

In [14]:
print("Seleccionando características (features) para el entrenamiento...")
cols_features = [
    'z_score', 'dist_ema288', 'vol_spike', 
    'lower_wick_ratio', 'upper_wick_ratio', 'body_ratio'
]
print(f"Características utilizadas: {cols_features}")

# Separación de X (features) e y (target)
X_train = df_is[cols_features].values
y_train = df_is['target'].values
X_test = df_oos[cols_features].values
y_test = df_oos['target'].values
print("Matrices de entrenamiento y validación generadas.")

print("Calculando balanceo de clases...")
sum_y = sum(y_train)
len_y = len(y_train)
print(f"  -> Total muestras entrenamiento: {len_y}")
print(f"  -> Casos Positivos (Target=1): {sum_y}")
print(f"  -> Casos Negativos (Target=0): {len_y - sum_y}")

scale_weight = (len_y - sum_y) / sum_y if sum_y > 0 else 1
scale_weight = min(scale_weight, 15)
print(f"Peso asignado a la clase positiva (scale_pos_weight): {scale_weight:.2f}")

Seleccionando características (features) para el entrenamiento...
Características utilizadas: ['z_score', 'dist_ema288', 'vol_spike', 'lower_wick_ratio', 'upper_wick_ratio', 'body_ratio']
Matrices de entrenamiento y validación generadas.
Calculando balanceo de clases...
  -> Total muestras entrenamiento: 62246
  -> Casos Positivos (Target=1): 14467
  -> Casos Negativos (Target=0): 47779
Peso asignado a la clase positiva (scale_pos_weight): 3.30


Llegó el momento de construir la arquitectura del modelo. Creamos un Pipeline de Scikit-Learn que primero estandariza los datos (StandardScaler) para que todas las métricas estén en la misma escala, y luego aplica el clasificador XGBoost con los hiperparámetros optimizados. Una vez configurado, procedemos a entrenarlo.

In [15]:
print("Construyendo el Pipeline del modelo (Escalado + XGBoost)...")
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('xgb', XGBClassifier(
        objective='binary:logistic',
        tree_method='hist',
        device='cuda', # Si tienes problemas con CUDA, cambia esto a 'cpu'
        n_estimators=700,
        learning_rate=0.02,
        max_depth=4,
        subsample=0.85,
        colsample_bytree=1.0,
        scale_pos_weight=scale_weight,
        random_state=42
    ))
])

print("Iniciando proceso de entrenamiento (Fit)...")
print("El modelo está aprendiendo patrones, esto puede tomar unos segundos...")
pipeline.fit(X_train, y_train)
print("¡Entrenamiento finalizado con éxito!")

print("Configurando el modelo entrenado para ejecutarse en CPU para la inferencia...")
pipeline.named_steps['xgb'].get_booster().set_param({'device': 'cpu'})

Construyendo el Pipeline del modelo (Escalado + XGBoost)...
Iniciando proceso de entrenamiento (Fit)...
El modelo está aprendiendo patrones, esto puede tomar unos segundos...


c:\Users\Sebas Uni\AppData\Local\Programs\Python\Python311\Lib\site-packages\xgboost\training.py:200: UserWarning: [16:11:33] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\context.cc:53: No visible GPU is found, setting device to CPU.
  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\Sebas Uni\AppData\Local\Programs\Python\Python311\Lib\site-packages\xgboost\training.py:200: UserWarning: [16:11:33] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\context.cc:207: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  bst.update(dtrain, iteration=i, fobj=obj)


¡Entrenamiento finalizado con éxito!
Configurando el modelo entrenado para ejecutarse en CPU para la inferencia...


Para comprobar que el modelo ha aprendido algo coherente antes de llevarlo al Backtest, realizamos una predicción rápida sobre el conjunto de datos Out-of-Sample y mostramos su precisión global. Finalmente, exportamos el Pipeline completo como un archivo .joblib para poder usarlo en el Cuaderno 3 o en producción.

In [16]:
print("Realizando predicción preliminar sobre el conjunto de Validación (OOS)...")
y_pred = pipeline.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Precisión global (Accuracy) en OOS: {accuracy * 100:.2f}%")

print("Preparando exportación del modelo...")
joblib.dump(pipeline, RUTA_MODELO)
print(f"¡Modelo guardado exitosamente en: {RUTA_MODELO}!")

print("\n" + "="*50)
print("  FIN DEL CUADERNO 2 - MODELO LISTO PARA BACKTEST  ")
print("="*50)

Realizando predicción preliminar sobre el conjunto de Validación (OOS)...
Precisión global (Accuracy) en OOS: 61.76%
Preparando exportación del modelo...
¡Modelo guardado exitosamente en: c:\Users\Sebas Uni\Desktop\AITradingIntegrated\models\EURUSD_M5_scalper.joblib!

  FIN DEL CUADERNO 2 - MODELO LISTO PARA BACKTEST  


In [19]:
import os
import onnxmltools
from skl2onnx.common.data_types import FloatTensorType
from skl2onnx import update_registered_converter, convert_sklearn
from skl2onnx.common.shape_calculator import calculate_linear_classifier_output_shapes
from xgboost import XGBClassifier
from onnxmltools.convert.xgboost.operator_converters.XGBoost import convert_xgboost

print("Registrando convertidores de XGBoost para el Pipeline...")

# Registramos el modelo utilizando el calculador compatible de scikit-learn
update_registered_converter(
    XGBClassifier,
    "XGBoostXGBClassifier",
    calculate_linear_classifier_output_shapes,
    convert_xgboost,
    options={"nocl": [True, False], "zipmap": [True, False, "columns"]}
)

print("Exportando modelo a formato ONNX...")

# Definimos las variables de entrada (tus 6 features)
initial_type = [('float_input', FloatTensorType([None, 6]))]

# Convertimos nuestro Pipeline completo a formato ONNX
onnx_model = convert_sklearn(
    pipeline, 
    initial_types=initial_type,
    target_opset={"": 12, "ai.onnx.ml": 2}
)

# Guardamos el archivo con la extensión solicitada
ruta_onnx = os.path.join(MODELS_DIR, f"{SYMBOL}_M5_scalper.onnx")
with open(ruta_onnx, "wb") as f:
    f.write(onnx_model.SerializeToString())

print(f"¡Modelo optimizado exportado con éxito en formato ONNX: {ruta_onnx}!")

Registrando convertidores de XGBoost para el Pipeline...
Exportando modelo a formato ONNX...
¡Modelo optimizado exportado con éxito en formato ONNX: c:\Users\Sebas Uni\Desktop\AITradingIntegrated\models\EURUSD_M5_scalper.onnx!
